# V8 — 01 Exploration

**Purpose.** Look at the real data before modelling anything, and let what we
see justify the choices made in Phases 2–6. Every section below ends with
*"what this means for the build"* — exploration that doesn't change a decision
is just sightseeing.

**Run order:** top to bottom. Restart the kernel and run all before committing,
so the saved output matches the code.

**Note on imports:** this notebook imports from `src/`. Nothing is copy-pasted
out of the pipeline — if the loader changes, this notebook changes with it.


In [ ]:
# --- Setup -------------------------------------------------------------------
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Notebooks live in notebooks/, so the repo root is one level up. Adding it to
# sys.path lets us import the real pipeline code instead of duplicating it.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from src.config import load_config
from src import ingest

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 50)

config = load_config(REPO_ROOT / "config.yaml")
print("config loaded, seed =", config.get("seed"))

In [ ]:
# --- Load the data -----------------------------------------------------------
# Same rule as the pipeline: real files from data/raw if present, else the
# bundled sample. The sample is tiny, so most numbers below will look odd on it.
raw_files = ingest.expected_raw_files(config)
found = [f for f in raw_files if f["exists"]]

if found:
    df = ingest.load_many(found, config)
    print(f"Loaded REAL data: {len(found)} file(s)")
else:
    df = ingest.load_matches(REPO_ROOT / config["data"]["sample_path"], config)
    print("Loaded the bundled SAMPLE — download the real files for meaningful numbers:")
    for f in raw_files:
        print("   ", f["url"])

# Parse dates once here so every section below can use them.
df["MatchDate"] = pd.to_datetime(df["Date"], dayfirst=True, format="mixed", errors="coerce")
df = df.sort_values("MatchDate").reset_index(drop=True)

print()
print(ingest.build_shape_report(df, config))

## 1. Home advantage

The single strongest effect in football, and the first parameter the Dixon-Coles
model has to capture. If the home/draw/away split here is far from the ~46/26/28
that top European leagues usually show, something is wrong with the data, not
with football.

In [ ]:
# --- Outcome split and goal means -------------------------------------------
outcomes = df["FTR"].value_counts(normalize=True).reindex(["H", "D", "A"]) * 100
print("Outcome split (%):")
for result, pct in outcomes.items():
    label = {"H": "home win", "D": "draw", "A": "away win"}[result]
    print(f"  {label:<9} {pct:5.1f}%")

home_goals, away_goals = df["FTHG"].mean(), df["FTAG"].mean()
print(f"\nMean goals — home {home_goals:.2f}, away {away_goals:.2f}")
print(f"Home scoring advantage: {home_goals / away_goals:.2f}x")

# Per season, to check the effect is stable rather than a one-season fluke.
if "Season" in df.columns:
    per_season = df.groupby("Season").agg(
        matches=("FTR", "size"),
        home_win_pct=("FTR", lambda s: (s == "H").mean() * 100),
        home_goals=("FTHG", "mean"),
        away_goals=("FTAG", "mean"),
    ).round(2)
    print()
    print(per_season)

In [ ]:
# --- Chart: outcome split ----------------------------------------------------
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(["Home win", "Draw", "Away win"], outcomes.values, color=["#2c7fb8", "#999999", "#d95f0e"])
ax.set_ylabel("% of matches")
ax.set_title("Match outcomes")
for i, v in enumerate(outcomes.values):
    ax.text(i, v + 0.6, f"{v:.1f}%", ha="center")
ax.set_ylim(0, max(outcomes.values) * 1.2)
plt.tight_layout()
plt.show()

**What this means for the build.** Home advantage is a single multiplicative
term in the Dixon-Coles likelihood (Phase 3). The per-season table is the check
that it's stable enough to estimate as one parameter rather than per season.

## 2. Are goals Poisson?

The whole model rests on treating goals as roughly Poisson. "Roughly" is the
important word: Dixon and Coles (1997) showed that independent Poisson
**underestimates** the frequency of 0-0, 1-0, 0-1 and 1-1, which is exactly why
their τ (tau) correction exists.

Below we compare what actually happened with what independent Poisson predicts.
If the four low scores come out above expectation, we've reproduced the finding
the model is built on — in our own data.

In [ ]:
# --- Observed vs independent-Poisson scorelines ------------------------------
from scipy.stats import poisson

max_goals = 5
obs = (
    df.groupby(["FTHG", "FTAG"]).size().rename("observed")
    .reset_index().query("FTHG <= @max_goals and FTAG <= @max_goals")
)

# Expected count if home and away goals were independent Poisson draws with the
# observed means. This is the null the tau correction departs from.
obs["expected"] = [
    poisson.pmf(h, home_goals) * poisson.pmf(a, away_goals) * len(df)
    for h, a in zip(obs["FTHG"], obs["FTAG"])
]
obs["diff_pct"] = (obs["observed"] / obs["expected"] - 1) * 100

low_scores = obs[(obs["FTHG"] <= 1) & (obs["FTAG"] <= 1)].copy()
low_scores["scoreline"] = low_scores["FTHG"].astype(str) + "-" + low_scores["FTAG"].astype(str)

print("The four scorelines Dixon-Coles corrects (tau):")
print(low_scores[["scoreline", "observed", "expected", "diff_pct"]]
      .round({"expected": 1, "diff_pct": 1}).to_string(index=False))

print("\nBiggest departures from independent Poisson overall:")
top = obs.reindex(obs["diff_pct"].abs().sort_values(ascending=False).index).head(6)
top = top.assign(scoreline=top["FTHG"].astype(str) + "-" + top["FTAG"].astype(str))
print(top[["scoreline", "observed", "expected", "diff_pct"]]
      .round({"expected": 1, "diff_pct": 1}).to_string(index=False))

**What this means for the build.** Record the four `diff_pct` values in
`PROGRESS.md`. When Phase 3 fits ρ (rho), its sign and size should be consistent
with what you just measured — that's a free sanity check on the fit, and a
strong thing to show a panel: *"I didn't take the correction on faith, I measured
the effect it exists to fix."*

## 3. What the market charges: the overround

Bookmaker odds don't sum to 100%. The excess is the **overround** (or vig) — the
bookmaker's margin. Before any odds can be treated as probabilities they must be
de-vigged, and §1 of the methodology still has that decision open
(proportional vs Shin).

We also compare the odds we'd bet into with the **closing** odds. The closing
line is the market's most informed price, which is why beating it (CLV) is the
project's headline metric.

In [ ]:
# --- Overround, opening vs closing -------------------------------------------
def overround(frame, cols):
    """Sum of implied probabilities minus 1, as a percentage.

    1/odds is the implied probability of each outcome. For a fair book they'd sum
    to 1.00; anything above is the bookmaker's margin.
    """
    present = [c for c in cols if c in frame.columns]
    if len(present) < len(cols):
        return None
    implied = 1 / frame[present]
    return (implied.sum(axis=1) - 1) * 100

bet_cols = config["schema"]["odds_columns"] if "schema" in config else ["B365H", "B365D", "B365A"]
close_cols = config["schema"]["closing_odds_columns"] if "schema" in config else ["PSCH", "PSCD", "PSCA"]

for label, cols in [("odds we bet into", bet_cols), ("closing odds", close_cols)]:
    margin = overround(df, cols)
    if margin is None:
        print(f"{label}: columns {cols} not in this data — skipped")
        continue
    print(f"{label} ({', '.join(cols)}):")
    print(f"   mean overround {margin.mean():.2f}%   median {margin.median():.2f}%"
          f"   min {margin.min():.2f}%   max {margin.max():.2f}%")

In [ ]:
# --- Is the market itself well calibrated? -----------------------------------
# Take the closing home-win price, de-vig it the simple (proportional) way, bucket
# the probabilities, and compare each bucket with how often home actually won.
# If the market is sharp, the two columns track each other closely — which is the
# bar our model has to clear.
if all(c in df.columns for c in close_cols):
    implied = 1 / df[close_cols]
    implied = implied.div(implied.sum(axis=1), axis=0)      # proportional de-vig
    p_home = implied[close_cols[0]]

    buckets = pd.cut(p_home, bins=np.arange(0, 1.01, 0.1))
    table = pd.DataFrame({
        "market_p_home": p_home,
        "home_won": (df["FTR"] == "H").astype(int),
        "bucket": buckets,
    })
    summary = table.groupby("bucket", observed=True).agg(
        matches=("home_won", "size"),
        market_says=("market_p_home", "mean"),
        actually_happened=("home_won", "mean"),
    ).round(3)
    print("Market calibration — closing home-win price vs reality:")
    print(summary.to_string())
else:
    print(f"Closing odds columns {close_cols} not present — skipped.")

**What this means for the build.** Two things. First, the overround gap
between the odds we bet into and the closing odds tells you how much margin a
value bet must overcome. Second, that calibration table is the benchmark for
Phase 4: a model whose probabilities are worse-calibrated than the closing line
has no edge, however clever it looks.

## 4. Shots as an expected-goals proxy

Phase 2 builds an xG proxy from shots, because goals are rare and noisy while
shots are plentiful. That only helps if shots actually carry signal.

In [ ]:
# --- Shots vs goals ----------------------------------------------------------
shot_cols = ["HS", "AS", "HST", "AST"]
missing = [c for c in shot_cols if c not in df.columns]

if missing:
    print(f"Shot columns missing: {missing} — skipped (expected on the sample).")
else:
    pairs = [("HS", "FTHG", "home shots"), ("HST", "FTHG", "home shots on target"),
             ("AS", "FTAG", "away shots"), ("AST", "FTAG", "away shots on target")]
    print("Correlation with goals scored (same match):")
    for shots, goals, label in pairs:
        r = df[shots].corr(df[goals])
        print(f"  {label:<24} r = {r:.3f}")

    # Conversion rate: how many shots on target become goals?
    conv_home = df["FTHG"].sum() / df["HST"].sum()
    conv_away = df["FTAG"].sum() / df["AST"].sum()
    print(f"\nGoals per shot on target — home {conv_home:.3f}, away {conv_away:.3f}")

    fig, ax = plt.subplots(figsize=(6, 4))
    jitter = np.random.default_rng(config.get("seed", 42)).normal(0, 0.08, len(df))
    ax.scatter(df["HST"], df["FTHG"] + jitter, s=8, alpha=0.3, color="#2c7fb8")
    ax.set_xlabel("Home shots on target")
    ax.set_ylabel("Home goals (jittered)")
    ax.set_title("Shots on target vs goals")
    plt.tight_layout()
    plt.show()

**What this means for the build.** A correlation around 0.5–0.6 for shots on
target is normal and is enough for a useful proxy — but notice it is far from 1,
which is the honest reason the model predicts a *distribution* of scorelines
rather than a scoreline.

## 5. Findings to record

Copy the numbers you actually got into `PROGRESS.md` under today's entry:

1. Outcome split and mean goals (home / away).
2. The four low-score `diff_pct` values — the evidence for the τ correction.
3. Mean overround for the odds you bet into and for the closing odds.
4. Whether the market's closing probabilities track reality in the bucket table.
5. Shots-on-target correlation with goals.

Each one is a prediction about what a later phase should produce. When Phase 3
or Phase 4 disagrees with what you measured here, that's a bug worth chasing —
which is the real reason this notebook exists.